# 03 Training Linear Model Using Colab CPU Resources

# ============================================================
# Colab-ready tokenization demo (character → word → subword)
# Works with ~8 GB RAM easily
# ============================================================

In [2]:
!pip install -q tokenizers   # only needed once per runtime
# Colab cell: installs the HuggingFace `tokenizers` lib for the BPE section.
# -q = quiet. Colab already ships torch/numpy; only tokenizers is missing.
# On a local machine use the venv/requirements.txt instead of pip-in-notebook.
# Colab cell: installs the HuggingFace `tokenizers` lib for the BPE section.
# -q = quiet. Colab already ships torch/numpy; only tokenizers is missing.
# On a local machine use the venv/requirements.txt instead of pip-in-notebook.
# Colab cell: installs the HuggingFace `tokenizers` lib for the BPE section.
# -q = quiet. Colab already ships torch/numpy; only tokenizers is missing.
# On a local machine use the venv/requirements.txt instead of pip-in-notebook.
# Colab cell: installs the HuggingFace `tokenizers` lib for the BPE section.
# -q = quiet. Colab already ships torch/numpy; only tokenizers is missing.
# On a local machine use the venv/requirements.txt instead of pip-in-notebook.
# Colab cell: installs the HuggingFace `tokenizers` lib for the BPE section.
# -q = quiet. Colab already ships torch/numpy; only tokenizers is missing.
# On a local machine use the venv/requirements.txt instead of pip-in-notebook.
# Colab cell: installs the HuggingFace `tokenizers` lib for the BPE section.
# -q = quiet. Colab already ships torch/numpy; only tokenizers is missing.
# On a local machine use the venv/requirements.txt instead of pip-in-notebook.

import torch
import torch.nn.functional as F
import math
import os
from collections import Counter
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
# HuggingFace tokenizers: Tokenizer = the pipeline wrapper; models.BPE = the
# byte-pair-encoding model; trainers.BpeTrainer learns merge rules;
# pre_tokenizers split text before training; decoders turn ids back to text.
# These classes implement the byte-level BPE (GPT-2 style) used in stages 04/06.
# HuggingFace tokenizers: Tokenizer = the pipeline wrapper; models.BPE = the
# byte-pair-encoding model; trainers.BpeTrainer learns merge rules;
# pre_tokenizers split text before training; decoders turn ids back to text.
# These classes implement the byte-level BPE (GPT-2 style) used in stages 04/06.
# HuggingFace tokenizers: Tokenizer = the pipeline wrapper; models.BPE = the
# byte-pair-encoding model; trainers.BpeTrainer learns merge rules;
# pre_tokenizers split text before training; decoders turn ids back to text.
# These classes implement the byte-level BPE (GPT-2 style) used in stages 04/06.
# HuggingFace tokenizers: Tokenizer = the pipeline wrapper; models.BPE = the
# byte-pair-encoding model; trainers.BpeTrainer learns merge rules;
# pre_tokenizers split text before training; decoders turn ids back to text.
# These classes implement the byte-level BPE (GPT-2 style) used in stages 04/06.
# HuggingFace tokenizers: Tokenizer = the pipeline wrapper; models.BPE = the
# byte-pair-encoding model; trainers.BpeTrainer learns merge rules;
# pre_tokenizers split text before training; decoders turn ids back to text.
# These classes implement the byte-level BPE (GPT-2 style) used in stages 04/06.
# HuggingFace tokenizers: Tokenizer = the pipeline wrapper; models.BPE = the
# byte-pair-encoding model; trainers.BpeTrainer learns merge rules;
# pre_tokenizers split text before training; decoders turn ids back to text.
# These classes implement the byte-level BPE (GPT-2 style) used in stages 04/06.
from tokenizers.processors import TemplateProcessing

# ------------------------------------------------------------
# 1. Download data (Tiny Shakespeare – classic starter dataset)
# ------------------------------------------------------------

In [3]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O input.txt
# Colab: fetch Tiny Shakespeare (~1.1 MB, karpathy's classic starter corpus).
# Locally this dataset already lives in dataset/input.txt (gitignored).
# Colab: fetch Tiny Shakespeare (~1.1 MB, karpathy's classic starter corpus).
# Locally this dataset already lives in dataset/input.txt (gitignored).
# Colab: fetch Tiny Shakespeare (~1.1 MB, karpathy's classic starter corpus).
# Locally this dataset already lives in dataset/input.txt (gitignored).
# Colab: fetch Tiny Shakespeare (~1.1 MB, karpathy's classic starter corpus).
# Locally this dataset already lives in dataset/input.txt (gitignored).
# Colab: fetch Tiny Shakespeare (~1.1 MB, karpathy's classic starter corpus).
# Locally this dataset already lives in dataset/input.txt (gitignored).
# Colab: fetch Tiny Shakespeare (~1.1 MB, karpathy's classic starter corpus).
# Locally this dataset already lives in dataset/input.txt (gitignored).

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()
# The whole corpus as one string. Every tokenization level below reads this.
# The whole corpus as one string. Every tokenization level below reads this.
# The whole corpus as one string. Every tokenization level below reads this.
# The whole corpus as one string. Every tokenization level below reads this.
# The whole corpus as one string. Every tokenization level below reads this.
# The whole corpus as one string. Every tokenization level below reads this.

print(f"Dataset length: {len(text):,} characters")
print("First 300 chars:\n", text[:300])
print("-" * 60)

Dataset length: 1,115,394 characters
First 300 chars:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us
------------------------------------------------------------


# ------------------------------------------------------------
# 2. Character-level (exactly what you already have)
# ------------------------------------------------------------

In [4]:
chars = sorted(list(set(text)))
# CHARACTER-LEVEL tokenization (baseline). sorted(set(...)) = deterministic
# vocab of every distinct character. len(chars) = 65 for this corpus.
# CHARACTER-LEVEL tokenization (baseline). sorted(set(...)) = deterministic
# vocab of every distinct character. len(chars) = 65 for this corpus.
# CHARACTER-LEVEL tokenization (baseline). sorted(set(...)) = deterministic
# vocab of every distinct character. len(chars) = 65 for this corpus.
# CHARACTER-LEVEL tokenization (baseline). sorted(set(...)) = deterministic
# vocab of every distinct character. len(chars) = 65 for this corpus.
# CHARACTER-LEVEL tokenization (baseline). sorted(set(...)) = deterministic
# vocab of every distinct character. len(chars) = 65 for this corpus.
# CHARACTER-LEVEL tokenization (baseline). sorted(set(...)) = deterministic
# vocab of every distinct character. len(chars) = 65 for this corpus.
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size_char = len(chars)

print(f"Character vocab size: {vocab_size_char}")
print("Some characters:", chars[:20])

# Encode whole text as character indices
char_data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
# The full corpus as one long tensor of char ids. dtype=long because these
# are lookup indices. Baseline stats: vocab=65 but sequences are huge
# (1,115,394 steps) -> long-range dependencies are hard to learn.
# The full corpus as one long tensor of char ids. dtype=long because these
# are lookup indices. Baseline stats: vocab=65 but sequences are huge
# (1,115,394 steps) -> long-range dependencies are hard to learn.
# The full corpus as one long tensor of char ids. dtype=long because these
# are lookup indices. Baseline stats: vocab=65 but sequences are huge
# (1,115,394 steps) -> long-range dependencies are hard to learn.
# The full corpus as one long tensor of char ids. dtype=long because these
# are lookup indices. Baseline stats: vocab=65 but sequences are huge
# (1,115,394 steps) -> long-range dependencies are hard to learn.
# The full corpus as one long tensor of char ids. dtype=long because these
# are lookup indices. Baseline stats: vocab=65 but sequences are huge
# (1,115,394 steps) -> long-range dependencies are hard to learn.
# The full corpus as one long tensor of char ids. dtype=long because these
# are lookup indices. Baseline stats: vocab=65 but sequences are huge
# (1,115,394 steps) -> long-range dependencies are hard to learn.
print(f"Character sequence length: {len(char_data):,}")

Character vocab size: 65
Some characters: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G']
Character sequence length: 1,115,394


# ------------------------------------------------------------
# 3. Word-level (simple split on whitespace + keep punctuation)
# ------------------------------------------------------------

In [5]:
# Very basic word tokenizer (good enough for learning)
import re
words = re.findall(r"\w+|[^\w\s]", text)   # words or single punctuation
# WORD-LEVEL tokenization. The regex keeps words OR any single punctuation
# char (so commas/periods become tokens instead of vanishing). Compare
# vocab sizes: ~7k words vs 65 chars — bigger vocab, ~4x shorter sequence.
# WORD-LEVEL tokenization. The regex keeps words OR any single punctuation
# char (so commas/periods become tokens instead of vanishing). Compare
# vocab sizes: ~7k words vs 65 chars — bigger vocab, ~4x shorter sequence.
# WORD-LEVEL tokenization. The regex keeps words OR any single punctuation
# char (so commas/periods become tokens instead of vanishing). Compare
# vocab sizes: ~7k words vs 65 chars — bigger vocab, ~4x shorter sequence.
# WORD-LEVEL tokenization. The regex keeps words OR any single punctuation
# char (so commas/periods become tokens instead of vanishing). Compare
# vocab sizes: ~7k words vs 65 chars — bigger vocab, ~4x shorter sequence.
# WORD-LEVEL tokenization. The regex keeps words OR any single punctuation
# char (so commas/periods become tokens instead of vanishing). Compare
# vocab sizes: ~7k words vs 65 chars — bigger vocab, ~4x shorter sequence.
# WORD-LEVEL tokenization. The regex keeps words OR any single punctuation
# char (so commas/periods become tokens instead of vanishing). Compare
# vocab sizes: ~7k words vs 65 chars — bigger vocab, ~4x shorter sequence.
word_counts = Counter(words)
# Keep only words that appear at least a few times (to keep vocab reasonable)
min_freq = 2
# Drop words seen once: shrinks vocab and reserves rare cases for <UNK>.
# Trade-off: smaller vocab/shorter embeddings, but more unknowns.
# Drop words seen once: shrinks vocab and reserves rare cases for <UNK>.
# Trade-off: smaller vocab/shorter embeddings, but more unknowns.
# Drop words seen once: shrinks vocab and reserves rare cases for <UNK>.
# Trade-off: smaller vocab/shorter embeddings, but more unknowns.
# Drop words seen once: shrinks vocab and reserves rare cases for <UNK>.
# Trade-off: smaller vocab/shorter embeddings, but more unknowns.
# Drop words seen once: shrinks vocab and reserves rare cases for <UNK>.
# Trade-off: smaller vocab/shorter embeddings, but more unknowns.
# Drop words seen once: shrinks vocab and reserves rare cases for <UNK>.
# Trade-off: smaller vocab/shorter embeddings, but more unknowns.
vocab_words = [w for w, c in word_counts.items() if c >= min_freq]
word_to_idx = {w: i for i, w in enumerate(vocab_words)}
idx_to_word = {i: w for w, i in word_to_idx.items()}
vocab_size_word = len(vocab_words)

print(f"\nWord vocab size (freq ≥ {min_freq}): {vocab_size_word}")
print("Most common words:", word_counts.most_common(10))

# Encode text as word indices (unknown words become a special <UNK>)
UNK = "<UNK>"
# UNK = placeholder id for out-of-vocabulary words at encode time.
# Character and byte-level BPE never need it (everything is representable),
# but a fixed word vocab always meets unseen words at generation time.
# UNK = placeholder id for out-of-vocabulary words at encode time.
# Character and byte-level BPE never need it (everything is representable),
# but a fixed word vocab always meets unseen words at generation time.
# UNK = placeholder id for out-of-vocabulary words at encode time.
# Character and byte-level BPE never need it (everything is representable),
# but a fixed word vocab always meets unseen words at generation time.
# UNK = placeholder id for out-of-vocabulary words at encode time.
# Character and byte-level BPE never need it (everything is representable),
# but a fixed word vocab always meets unseen words at generation time.
# UNK = placeholder id for out-of-vocabulary words at encode time.
# Character and byte-level BPE never need it (everything is representable),
# but a fixed word vocab always meets unseen words at generation time.
# UNK = placeholder id for out-of-vocabulary words at encode time.
# Character and byte-level BPE never need it (everything is representable),
# but a fixed word vocab always meets unseen words at generation time.
if UNK not in word_to_idx:
    word_to_idx[UNK] = vocab_size_word
    idx_to_word[vocab_size_word] = UNK
    vocab_size_word += 1

word_data = []
for w in words:
    word_data.append(word_to_idx.get(w, word_to_idx[UNK]))
# .get(w, UNK-id) = the fallback: rare/unseen word -> <UNK> index.
# .get(w, UNK-id) = the fallback: rare/unseen word -> <UNK> index.
# .get(w, UNK-id) = the fallback: rare/unseen word -> <UNK> index.
# .get(w, UNK-id) = the fallback: rare/unseen word -> <UNK> index.
# .get(w, UNK-id) = the fallback: rare/unseen word -> <UNK> index.
# .get(w, UNK-id) = the fallback: rare/unseen word -> <UNK> index.
word_data = torch.tensor(word_data, dtype=torch.long)
print(f"Word sequence length: {len(word_data):,}")


Word vocab size (freq ≥ 2): 7286
Most common words: [(',', 19846), (':', 10316), ('.', 7885), ("'", 6187), ('the', 5442), ('I', 5043), ('to', 4112), ('and', 3763), (';', 3628), ('of', 3314)]
Word sequence length: 262,927


# ------------------------------------------------------------
# 4. Subword-level (BPE – the modern standard)
# ------------------------------------------------------------

In [6]:
# Train a tiny BPE tokenizer on the same text
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
# SUBWORD-LEVEL: BPE starts from characters and repeatedly merges the most
# frequent adjacent pair into a new token — vocab size is a dial you pick.
# Middle ground: seq shorter than chars, vocab far smaller than words.
# SUBWORD-LEVEL: BPE starts from characters and repeatedly merges the most
# frequent adjacent pair into a new token — vocab size is a dial you pick.
# Middle ground: seq shorter than chars, vocab far smaller than words.
# SUBWORD-LEVEL: BPE starts from characters and repeatedly merges the most
# frequent adjacent pair into a new token — vocab size is a dial you pick.
# Middle ground: seq shorter than chars, vocab far smaller than words.
# SUBWORD-LEVEL: BPE starts from characters and repeatedly merges the most
# frequent adjacent pair into a new token — vocab size is a dial you pick.
# Middle ground: seq shorter than chars, vocab far smaller than words.
# SUBWORD-LEVEL: BPE starts from characters and repeatedly merges the most
# frequent adjacent pair into a new token — vocab size is a dial you pick.
# Middle ground: seq shorter than chars, vocab far smaller than words.
# SUBWORD-LEVEL: BPE starts from characters and repeatedly merges the most
# frequent adjacent pair into a new token — vocab size is a dial you pick.
# Middle ground: seq shorter than chars, vocab far smaller than words.
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()   # split on spaces first
# Pre-tokenizer runs BEFORE training/encoding: splits on whitespace so
# merges never cross word boundaries.
# Pre-tokenizer runs BEFORE training/encoding: splits on whitespace so
# merges never cross word boundaries.
# Pre-tokenizer runs BEFORE training/encoding: splits on whitespace so
# merges never cross word boundaries.
# Pre-tokenizer runs BEFORE training/encoding: splits on whitespace so
# merges never cross word boundaries.
# Pre-tokenizer runs BEFORE training/encoding: splits on whitespace so
# merges never cross word boundaries.
# Pre-tokenizer runs BEFORE training/encoding: splits on whitespace so
# merges never cross word boundaries.
trainer = trainers.BpeTrainer(
    vocab_size=1000,          # small for demo – real models use 8k–50k
# vocab_size=1000 is a demo dial; GPT-2 uses ~50k. Bigger vocab = shorter
# sequences but more parameters in the output head (logits = vocab-sized).
# vocab_size=1000 is a demo dial; GPT-2 uses ~50k. Bigger vocab = shorter
# sequences but more parameters in the output head (logits = vocab-sized).
# vocab_size=1000 is a demo dial; GPT-2 uses ~50k. Bigger vocab = shorter
# sequences but more parameters in the output head (logits = vocab-sized).
# vocab_size=1000 is a demo dial; GPT-2 uses ~50k. Bigger vocab = shorter
# sequences but more parameters in the output head (logits = vocab-sized).
# vocab_size=1000 is a demo dial; GPT-2 uses ~50k. Bigger vocab = shorter
# sequences but more parameters in the output head (logits = vocab-sized).
# vocab_size=1000 is a demo dial; GPT-2 uses ~50k. Bigger vocab = shorter
# sequences but more parameters in the output head (logits = vocab-sized).
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"]
# Special tokens: [UNK] unknown, [PAD] batch padding, [BOS]/[EOS]
# sequence start/end. Reserved ids the trainer will never merge away.
# Special tokens: [UNK] unknown, [PAD] batch padding, [BOS]/[EOS]
# sequence start/end. Reserved ids the trainer will never merge away.
# Special tokens: [UNK] unknown, [PAD] batch padding, [BOS]/[EOS]
# sequence start/end. Reserved ids the trainer will never merge away.
# Special tokens: [UNK] unknown, [PAD] batch padding, [BOS]/[EOS]
# sequence start/end. Reserved ids the trainer will never merge away.
# Special tokens: [UNK] unknown, [PAD] batch padding, [BOS]/[EOS]
# sequence start/end. Reserved ids the trainer will never merge away.
# Special tokens: [UNK] unknown, [PAD] batch padding, [BOS]/[EOS]
# sequence start/end. Reserved ids the trainer will never merge away.
)
tokenizer.train_from_iterator([text], trainer=trainer)

# Optional: make decoding nicer
tokenizer.decoder = decoders.BPEDecoder()
# Decoder maps ids back to text for generation. NOTE (revision): the
# Whitespace pre-tokenizer + plain BPEDecoder combo loses exact spacing
# (see the squashed BPE sample below) — the byte-level BPE in stage 04/06
# fixes this by mapping bytes to printable chars and reversing exactly.
# Decoder maps ids back to text for generation. NOTE (revision): the
# Whitespace pre-tokenizer + plain BPEDecoder combo loses exact spacing
# (see the squashed BPE sample below) — the byte-level BPE in stage 04/06
# fixes this by mapping bytes to printable chars and reversing exactly.
# Decoder maps ids back to text for generation. NOTE (revision): the
# Whitespace pre-tokenizer + plain BPEDecoder combo loses exact spacing
# (see the squashed BPE sample below) — the byte-level BPE in stage 04/06
# fixes this by mapping bytes to printable chars and reversing exactly.
# Decoder maps ids back to text for generation. NOTE (revision): the
# Whitespace pre-tokenizer + plain BPEDecoder combo loses exact spacing
# (see the squashed BPE sample below) — the byte-level BPE in stage 04/06
# fixes this by mapping bytes to printable chars and reversing exactly.
# Decoder maps ids back to text for generation. NOTE (revision): the
# Whitespace pre-tokenizer + plain BPEDecoder combo loses exact spacing
# (see the squashed BPE sample below) — the byte-level BPE in stage 04/06
# fixes this by mapping bytes to printable chars and reversing exactly.
# Decoder maps ids back to text for generation. NOTE (revision): the
# Whitespace pre-tokenizer + plain BPEDecoder combo loses exact spacing
# (see the squashed BPE sample below) — the byte-level BPE in stage 04/06
# fixes this by mapping bytes to printable chars and reversing exactly.

# Save so you can reload later
tokenizer.save("shakespeare_bpe.json")
print("\nBPE tokenizer trained and saved as shakespeare_bpe.json")

# Encode the whole text with subword tokens
encoded = tokenizer.encode(text)
# Re-encode the corpus with the learned merges. Watch the compression
# number printed below: ~1.1M chars -> ~384k tokens (~2.9 chars/token).
# Re-encode the corpus with the learned merges. Watch the compression
# number printed below: ~1.1M chars -> ~384k tokens (~2.9 chars/token).
# Re-encode the corpus with the learned merges. Watch the compression
# number printed below: ~1.1M chars -> ~384k tokens (~2.9 chars/token).
# Re-encode the corpus with the learned merges. Watch the compression
# number printed below: ~1.1M chars -> ~384k tokens (~2.9 chars/token).
# Re-encode the corpus with the learned merges. Watch the compression
# number printed below: ~1.1M chars -> ~384k tokens (~2.9 chars/token).
# Re-encode the corpus with the learned merges. Watch the compression
# number printed below: ~1.1M chars -> ~384k tokens (~2.9 chars/token).
subword_ids = torch.tensor(encoded.ids, dtype=torch.long)
vocab_size_sub = tokenizer.get_vocab_size()

print(f"Subword (BPE) vocab size: {vocab_size_sub}")
print(f"Subword sequence length: {len(subword_ids):,}")
print("Example encoding of first sentence:")
print(tokenizer.encode(text[:100]).tokens)


BPE tokenizer trained and saved as shakespeare_bpe.json
Subword (BPE) vocab size: 1000
Subword sequence length: 383,663
Example encoding of first sentence:
['First', 'Citizen', ':', 'Be', 'fore', 'we', 'pro', 'ce', 'ed', 'any', 'f', 'ur', 'ther', ',', 'hear', 'me', 'speak', '.', 'All', ':', 'S', 'pe', 'ak', ',', 'speak', '.', 'First', 'Citizen', ':', 'You']


# ------------------------------------------------------------
# 5. Simple bigram model that works with ANY of the three
# ------------------------------------------------------------

In [7]:
def build_bigram_probs(data, vocab_size):
# Bigram model that works for ANY tokenization: the same count-and-
# normalize math from stage 01, now dtype=float32 tensors.
# Bigram model that works for ANY tokenization: the same count-and-
# normalize math from stage 01, now dtype=float32 tensors.
# Bigram model that works for ANY tokenization: the same count-and-
# normalize math from stage 01, now dtype=float32 tensors.
# Bigram model that works for ANY tokenization: the same count-and-
# normalize math from stage 01, now dtype=float32 tensors.
# Bigram model that works for ANY tokenization: the same count-and-
# normalize math from stage 01, now dtype=float32 tensors.
# Bigram model that works for ANY tokenization: the same count-and-
# normalize math from stage 01, now dtype=float32 tensors.
    """Count consecutive pairs and turn into probabilities."""
    counts = torch.zeros((vocab_size, vocab_size), dtype=torch.float32)
# counts[i, j] = how often token i is followed by token j. For the word
# vocab (7k+) this is 7286^2 ≈ 53M cells — fine, but note N-gram tables
# grow exponentially with context length (that's why we need neural nets).
# counts[i, j] = how often token i is followed by token j. For the word
# vocab (7k+) this is 7286^2 ≈ 53M cells — fine, but note N-gram tables
# grow exponentially with context length (that's why we need neural nets).
# counts[i, j] = how often token i is followed by token j. For the word
# vocab (7k+) this is 7286^2 ≈ 53M cells — fine, but note N-gram tables
# grow exponentially with context length (that's why we need neural nets).
# counts[i, j] = how often token i is followed by token j. For the word
# vocab (7k+) this is 7286^2 ≈ 53M cells — fine, but note N-gram tables
# grow exponentially with context length (that's why we need neural nets).
# counts[i, j] = how often token i is followed by token j. For the word
# vocab (7k+) this is 7286^2 ≈ 53M cells — fine, but note N-gram tables
# grow exponentially with context length (that's why we need neural nets).
# counts[i, j] = how often token i is followed by token j. For the word
# vocab (7k+) this is 7286^2 ≈ 53M cells — fine, but note N-gram tables
# grow exponentially with context length (that's why we need neural nets).
    for i in range(len(data) - 1):
        counts[data[i], data[i + 1]] += 1
    # Normalize rows → probability distribution
    row_sums = counts.sum(dim=1, keepdim=True)
    probs = counts / row_sums.clamp(min=1e-8)   # avoid division by zero
# clamp(min=1e-8) = the div-by-zero guard. Cleaner than the NaN-replace
# pass in stage 01/02: empty rows just become all-zeros directly.
# clamp(min=1e-8) = the div-by-zero guard. Cleaner than the NaN-replace
# pass in stage 01/02: empty rows just become all-zeros directly.
# clamp(min=1e-8) = the div-by-zero guard. Cleaner than the NaN-replace
# pass in stage 01/02: empty rows just become all-zeros directly.
# clamp(min=1e-8) = the div-by-zero guard. Cleaner than the NaN-replace
# pass in stage 01/02: empty rows just become all-zeros directly.
# clamp(min=1e-8) = the div-by-zero guard. Cleaner than the NaN-replace
# pass in stage 01/02: empty rows just become all-zeros directly.
# clamp(min=1e-8) = the div-by-zero guard. Cleaner than the NaN-replace
# pass in stage 01/02: empty rows just become all-zeros directly.
    return probs

# Build for each level (you can comment out the ones you don’t need)
print("\nBuilding bigram probability matrices…")
char_probs   = build_bigram_probs(char_data,   vocab_size_char)
word_probs   = build_bigram_probs(word_data,   vocab_size_word)
subword_probs = build_bigram_probs(subword_ids, vocab_size_sub)
# One probability table per tokenization level; generate() below samples
# from whichever table it is handed.
# One probability table per tokenization level; generate() below samples
# from whichever table it is handed.
# One probability table per tokenization level; generate() below samples
# from whichever table it is handed.
# One probability table per tokenization level; generate() below samples
# from whichever table it is handed.
# One probability table per tokenization level; generate() below samples
# from whichever table it is handed.
# One probability table per tokenization level; generate() below samples
# from whichever table it is handed.
print("Done.")


Building bigram probability matrices…
Done.


# ------------------------------------------------------------
# 6. Sampling / generation helper
# ------------------------------------------------------------

In [8]:
def generate(probs, idx_to_token, start_token, length=200, temperature=1.0):
# Tokenization-agnostic sampler: bigram chain rule, one token at a time.
# temperature is the new idea here (used again in stages 05/06).
# Tokenization-agnostic sampler: bigram chain rule, one token at a time.
# temperature is the new idea here (used again in stages 05/06).
# Tokenization-agnostic sampler: bigram chain rule, one token at a time.
# temperature is the new idea here (used again in stages 05/06).
# Tokenization-agnostic sampler: bigram chain rule, one token at a time.
# temperature is the new idea here (used again in stages 05/06).
# Tokenization-agnostic sampler: bigram chain rule, one token at a time.
# temperature is the new idea here (used again in stages 05/06).
# Tokenization-agnostic sampler: bigram chain rule, one token at a time.
# temperature is the new idea here (used again in stages 05/06).
    """Generate a sequence by sampling from the bigram table."""
    # Convert start token → index
    if isinstance(start_token, str):
        # try to find it
        start_idx = None
        for i, t in idx_to_token.items():
            if t == start_token:
                start_idx = i
                break
        if start_idx is None:
            start_idx = 0
    else:
        start_idx = start_token

    current = start_idx
    generated = [idx_to_token[current]]

    for _ in range(length - 1):
        p = probs[current]
        # optional temperature (softens / sharpens the distribution)
        if temperature != 1.0:
# TEMPERATURE: rescales the distribution before sampling.
#   t < 1 -> raise probs to 1/t > 1 power -> sharpens (safe, repetitive)
#   t > 1 -> flattens (creative, chaotic). t=1 = raw distribution.
# REVISION NOTE: exponentiating before renormalizing is the standard
# shortcut for a bigram table; the equivalent used on LOGITS in
# stages 05/06 is softmax(logits / t).
# TEMPERATURE: rescales the distribution before sampling.
#   t < 1 -> raise probs to 1/t > 1 power -> sharpens (safe, repetitive)
#   t > 1 -> flattens (creative, chaotic). t=1 = raw distribution.
# REVISION NOTE: exponentiating before renormalizing is the standard
# shortcut for a bigram table; the equivalent used on LOGITS in
# stages 05/06 is softmax(logits / t).
# TEMPERATURE: rescales the distribution before sampling.
#   t < 1 -> raise probs to 1/t > 1 power -> sharpens (safe, repetitive)
#   t > 1 -> flattens (creative, chaotic). t=1 = raw distribution.
# REVISION NOTE: exponentiating before renormalizing is the standard
# shortcut for a bigram table; the equivalent used on LOGITS in
# stages 05/06 is softmax(logits / t).
# TEMPERATURE: rescales the distribution before sampling.
#   t < 1 -> raise probs to 1/t > 1 power -> sharpens (safe, repetitive)
#   t > 1 -> flattens (creative, chaotic). t=1 = raw distribution.
# REVISION NOTE: exponentiating before renormalizing is the standard
# shortcut for a bigram table; the equivalent used on LOGITS in
# stages 05/06 is softmax(logits / t).
# TEMPERATURE: rescales the distribution before sampling.
#   t < 1 -> raise probs to 1/t > 1 power -> sharpens (safe, repetitive)
#   t > 1 -> flattens (creative, chaotic). t=1 = raw distribution.
# REVISION NOTE: exponentiating before renormalizing is the standard
# shortcut for a bigram table; the equivalent used on LOGITS in
# stages 05/06 is softmax(logits / t).
# TEMPERATURE: rescales the distribution before sampling.
#   t < 1 -> raise probs to 1/t > 1 power -> sharpens (safe, repetitive)
#   t > 1 -> flattens (creative, chaotic). t=1 = raw distribution.
# REVISION NOTE: exponentiating before renormalizing is the standard
# shortcut for a bigram table; the equivalent used on LOGITS in
# stages 05/06 is softmax(logits / t).
            p = p ** (1.0 / temperature)
            p = p / p.sum()
        next_idx = torch.multinomial(p, 1).item()
# torch.multinomial = sample an index with prob proportional to p
# (NumPy's np.random.choice(p=...) twin from stage 01).
# torch.multinomial = sample an index with prob proportional to p
# (NumPy's np.random.choice(p=...) twin from stage 01).
# torch.multinomial = sample an index with prob proportional to p
# (NumPy's np.random.choice(p=...) twin from stage 01).
# torch.multinomial = sample an index with prob proportional to p
# (NumPy's np.random.choice(p=...) twin from stage 01).
# torch.multinomial = sample an index with prob proportional to p
# (NumPy's np.random.choice(p=...) twin from stage 01).
# torch.multinomial = sample an index with prob proportional to p
# (NumPy's np.random.choice(p=...) twin from stage 01).
        generated.append(idx_to_token[next_idx])
        current = next_idx
    return generated

# ------------------------------------------------------------
# 7. Generate examples
# ------------------------------------------------------------

In [9]:
print("\n" + "="*60)
print("CHARACTER-LEVEL sample:")
print("".join(generate(char_probs, idx_to_char, start_token="A", length=200)))
# CHARACTER sample: locally plausible, globally nonsense — 1 char of
# context is simply too little (same lesson as stage 01).
# CHARACTER sample: locally plausible, globally nonsense — 1 char of
# context is simply too little (same lesson as stage 01).
# CHARACTER sample: locally plausible, globally nonsense — 1 char of
# context is simply too little (same lesson as stage 01).
# CHARACTER sample: locally plausible, globally nonsense — 1 char of
# context is simply too little (same lesson as stage 01).
# CHARACTER sample: locally plausible, globally nonsense — 1 char of
# context is simply too little (same lesson as stage 01).
# CHARACTER sample: locally plausible, globally nonsense — 1 char of
# context is simply too little (same lesson as stage 01).

print("\n" + "="*60)
print("WORD-LEVEL sample:")
print(" ".join(generate(word_probs, idx_to_word, start_token="The", length=50)))
# WORD sample: grammatical-ish because whole words carry meaning, but
# topics drift and <UNK> appears — 1 word of context still too little.
# WORD sample: grammatical-ish because whole words carry meaning, but
# topics drift and <UNK> appears — 1 word of context still too little.
# WORD sample: grammatical-ish because whole words carry meaning, but
# topics drift and <UNK> appears — 1 word of context still too little.
# WORD sample: grammatical-ish because whole words carry meaning, but
# topics drift and <UNK> appears — 1 word of context still too little.
# WORD sample: grammatical-ish because whole words carry meaning, but
# topics drift and <UNK> appears — 1 word of context still too little.
# WORD sample: grammatical-ish because whole words carry meaning, but
# topics drift and <UNK> appears — 1 word of context still too little.

print("\n" + "="*60)
print("SUBWORD-LEVEL (BPE) sample:")
# For BPE we use the tokenizer’s id_to_token
id_to_token = {v: k for k, v in tokenizer.get_vocab().items()}
tokens = generate(subword_probs, id_to_token, start_token="The", length=80)
print(tokenizer.decode([tokenizer.token_to_id(t) for t in tokens if t in tokenizer.get_vocab()]))# BPE sample: readable word-pieces but squashed spacing — the whitespace
# pre-tokenizer dropped spacing info (fixed by byte-level BPE in 04/06).
# The `if t in vocab` filter also silently drops sampled special ids.
# BPE sample: readable word-pieces but squashed spacing — the whitespace
# pre-tokenizer dropped spacing info (fixed by byte-level BPE in 04/06).
# The `if t in vocab` filter also silently drops sampled special ids.
# BPE sample: readable word-pieces but squashed spacing — the whitespace
# pre-tokenizer dropped spacing info (fixed by byte-level BPE in 04/06).
# The `if t in vocab` filter also silently drops sampled special ids.
# BPE sample: readable word-pieces but squashed spacing — the whitespace
# pre-tokenizer dropped spacing info (fixed by byte-level BPE in 04/06).
# The `if t in vocab` filter also silently drops sampled special ids.
# BPE sample: readable word-pieces but squashed spacing — the whitespace
# pre-tokenizer dropped spacing info (fixed by byte-level BPE in 04/06).
# The `if t in vocab` filter also silently drops sampled special ids.
# BPE sample: readable word-pieces but squashed spacing — the whitespace
# pre-tokenizer dropped spacing info (fixed by byte-level BPE in 04/06).
# The `if t in vocab` filter also silently drops sampled special ids.



CHARACTER-LEVEL sample:
An lor. tayiney mat leen, bitty ad tcee s grabengen h ou use; htnt s hinng M:
TEatomeadathe.
Tis grerey ica;

Sothexpimppaghesinethe w, tcolan o hee mas nde thecancuronor s l
I'l ggl h f mmaleedinl al

WORD-LEVEL sample:
The breath to depose The which his <UNK> the skin of York and cried , hearing how she within being Richard , had deserved no more . DUKE VINCENTIO : Well met , Which , and with trouble thee ; And , or two worthy man live in any thing

SUBWORD-LEVEL (BPE) sample:
Thedonotanyunderstandmelionta,come.Corwherebloodisbut'llbe,Iwerenoneelsebetheawear--ROMEO:Whatwasantigardofnatureisbemercy:ifhellundertakeherdwell,andafterlookuponhimthatsheep,arethycompletheethethandwithnoblance.
